# Section 6: Evaluation - Knowing if AI Actually Works

## Learning Objectives

By the end of this section, you will be able to:

1. **Choose appropriate evaluation methods** for different AI types (ML, LLM, RAG, Agent)
2. **Apply LLM-as-Judge patterns** while understanding their limitations
3. **Design human evaluation rubrics** that produce consistent ratings
4. **Plan statistically valid A/B tests** for AI features
5. **Build feedback loops** that continuously improve AI quality

---

**Time:** 45 minutes

**Format:** Interactive demos with discussion

In [ ]:
#@title Setup - Run this cell first (click play button)
%%capture
!pip install "ipywidgets>=7,<8" plotly numpy scipy

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from google.colab import output
output.enable_custom_widget_manager()

np.random.seed(42)
print("Setup complete! You're ready to explore AI evaluation.")

---

## The $50M Lesson: Meta's Galactica

In November 2022, Meta launched **Galactica** - an LLM trained on 48 million scientific papers, designed to help researchers.

**What happened?**
- Researchers quickly found it generated convincing-but-false citations
- It invented paper titles, author names, and DOIs that didn't exist
- It confidently stated incorrect scientific "facts"
- Twitter erupted with examples of dangerous misinformation

**The result:** Meta shut down Galactica after just **3 days**.

### The PM Question

> "How did a model trained on real papers generate fake citations?"

**The answer:** They evaluated for *fluency* (sounds good) but not *factuality* (is true).

**The lesson:** What you measure determines what you get. Galactica was excellent at sounding scientific - because that's what they tested for.

---

## Demo 1: Evaluation Dimensions Explorer

Different AI outputs need different evaluation criteria. What matters for a customer service chatbot is different from what matters for a code generator.

**Try this:** Select different use cases and see how the evaluation priorities change.

In [ ]:
#@title Evaluation Dimensions by Use Case

# Define evaluation dimensions and their importance by use case
dimensions = ['Accuracy', 'Relevance', 'Helpfulness', 'Clarity', 'Tone', 'Completeness', 'Creativity', 'Safety']

use_case_weights = {
    'Customer Service Bot': [0.7, 0.9, 0.9, 0.8, 0.9, 0.6, 0.2, 0.8],
    'Content Generator': [0.6, 0.8, 0.7, 0.8, 0.6, 0.7, 0.9, 0.6],
    'Code Assistant': [0.95, 0.9, 0.8, 0.7, 0.3, 0.8, 0.4, 0.5],
    'Research Summarizer': [0.9, 0.85, 0.7, 0.9, 0.4, 0.85, 0.3, 0.5],
    'Medical Triage Bot': [0.95, 0.9, 0.8, 0.9, 0.7, 0.9, 0.1, 0.99]
}

output_area = widgets.Output()

def show_dimensions(use_case):
    with output_area:
        clear_output(wait=True)
        weights = use_case_weights[use_case]
        
        # Create radar chart
        fig = go.Figure()
        
        fig.add_trace(go.Scatterpolar(
            r=weights + [weights[0]],  # Close the polygon
            theta=dimensions + [dimensions[0]],
            fill='toself',
            fillcolor='rgba(64, 184, 166, 0.3)',
            line=dict(color='#40B8A6', width=2),
            name=use_case
        ))
        
        fig.update_layout(
            polar=dict(
                radialaxis=dict(visible=True, range=[0, 1])
            ),
            title=f'Evaluation Priority for: {use_case}',
            showlegend=False,
            height=450
        )
        
        fig.show()
        
        # Show top 3 priorities
        sorted_dims = sorted(zip(dimensions, weights), key=lambda x: x[1], reverse=True)
        print(f"\nTop 3 Priorities for {use_case}:")
        for i, (dim, weight) in enumerate(sorted_dims[:3], 1):
            print(f"  {i}. {dim} ({weight*100:.0f}% importance)")

use_case_dropdown = widgets.Dropdown(
    options=list(use_case_weights.keys()),
    value='Customer Service Bot',
    description='Use Case:'
)

widgets.interactive(show_dimensions, use_case=use_case_dropdown)
display(use_case_dropdown, output_area)
show_dimensions('Customer Service Bot')

### PM Insight: Evaluation Priorities

Notice how **Safety** jumps to 99% for a Medical Triage Bot but only 50% for a Code Assistant. **Creativity** is high for Content Generation but should be near zero for medical applications.

> **Key Decision:** Before building any AI feature, define your evaluation dimensions and their relative importance. This drives what you test and what you optimize for.

---

## Demo 2: Inter-Rater Reliability Calculator

When humans evaluate AI outputs, how do you know if they're consistent? **Fleiss' Kappa** measures agreement between multiple raters.

**Try this:** Adjust the rater agreement levels and see how it affects reliability scores.

In [ ]:
#@title Inter-Rater Reliability Simulator

output_area2 = widgets.Output()

def calculate_reliability(agreement_pct, num_raters, num_items):
    with output_area2:
        clear_output(wait=True)
        
        # Simulate ratings based on agreement percentage
        np.random.seed(42)
        agreement_rate = agreement_pct / 100
        
        # Create simulated ratings (5-point scale)
        true_ratings = np.random.randint(1, 6, num_items)
        ratings = np.zeros((num_items, num_raters))
        
        for i in range(num_items):
            for j in range(num_raters):
                if np.random.random() < agreement_rate:
                    ratings[i, j] = true_ratings[i]
                else:
                    # Random deviation
                    ratings[i, j] = np.clip(true_ratings[i] + np.random.randint(-2, 3), 1, 5)
        
        # Calculate simplified Kappa approximation
        observed_agreement = agreement_rate
        expected_agreement = 0.2  # Random chance for 5 categories
        kappa = (observed_agreement - expected_agreement) / (1 - expected_agreement)
        kappa = max(-1, min(1, kappa))  # Bound between -1 and 1
        
        # Interpretation
        if kappa < 0:
            interpretation = "Poor - Worse than random"
            color = "#dc2626"
        elif kappa < 0.2:
            interpretation = "Slight - Barely better than chance"
            color = "#dc2626"
        elif kappa < 0.4:
            interpretation = "Fair - Need clearer guidelines"
            color = "#f59e0b"
        elif kappa < 0.6:
            interpretation = "Moderate - Acceptable for some uses"
            color = "#f59e0b"
        elif kappa < 0.8:
            interpretation = "Substantial - Good reliability"
            color = "#22c55e"
        else:
            interpretation = "Almost Perfect - Excellent agreement"
            color = "#22c55e"
        
        # Create gauge chart
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=kappa,
            domain={'x': [0, 1], 'y': [0, 1]},
            title={'text': "Fleiss' Kappa"},
            gauge={
                'axis': {'range': [-0.2, 1], 'tickwidth': 1},
                'bar': {'color': color},
                'steps': [
                    {'range': [-0.2, 0.2], 'color': '#fee2e2'},
                    {'range': [0.2, 0.4], 'color': '#fef3c7'},
                    {'range': [0.4, 0.6], 'color': '#fef9c3'},
                    {'range': [0.6, 0.8], 'color': '#d1fae5'},
                    {'range': [0.8, 1.0], 'color': '#a7f3d0'}
                ],
                'threshold': {
                    'line': {'color': "#1A3D4D", 'width': 4},
                    'thickness': 0.75,
                    'value': 0.6
                }
            }
        ))
        
        fig.update_layout(height=350)
        fig.show()
        
        print(f"\nInterpretation: {interpretation}")
        print(f"\nSetup: {num_raters} raters evaluated {num_items} items")
        print(f"Target agreement rate: {agreement_pct}%")
        print(f"\nNote: Kappa > 0.6 (dashed line) is typically required for reliable human evaluation.")

agreement_slider = widgets.IntSlider(value=70, min=20, max=95, step=5, description='Agreement %:')
raters_slider = widgets.IntSlider(value=3, min=2, max=5, step=1, description='# Raters:')
items_slider = widgets.IntSlider(value=50, min=20, max=200, step=10, description='# Items:')

ui = widgets.VBox([agreement_slider, raters_slider, items_slider])
out = widgets.interactive_output(calculate_reliability, {
    'agreement_pct': agreement_slider,
    'num_raters': raters_slider,
    'num_items': items_slider
})

display(ui, output_area2)
calculate_reliability(70, 3, 50)

### PM Insight: When Human Evaluation Fails

If your Kappa is below 0.6, your evaluation rubric needs work. Common fixes:

1. **Add concrete examples** for each rating level
2. **Reduce the scale** (3 points instead of 5)
3. **Train raters together** on edge cases
4. **Use pairwise comparison** ("Which is better: A or B?")

> **Key Decision:** Never launch human evaluation without first measuring inter-rater reliability on a pilot set.

---

## Discussion Break

**Turn to your neighbor and discuss:**

1. For your current AI project, what are the top 3 evaluation dimensions?
2. How would you measure each one? (Human eval? Automated metrics? Both?)
3. What's the minimum acceptable quality level for launch?

---

## Demo 3: LLM-as-Judge Simulator

Using an LLM to evaluate another LLM's output is increasingly common. But LLM judges have biases you need to understand.

**Try this:** See how different outputs get scored, and notice the biases.

In [ ]:
#@title LLM-as-Judge Bias Demonstration

# Sample outputs demonstrating different biases
sample_outputs = {
    'Concise Answer': {
        'text': 'The capital of France is Paris.',
        'human_score': 5,
        'llm_score': 3,
        'bias': 'Length Bias - LLMs often prefer longer responses'
    },
    'Verbose Answer': {
        'text': 'The capital of France is Paris, which is a beautiful city located in the north-central part of the country. Paris is known for its iconic Eiffel Tower, world-renowned museums like the Louvre, and rich cultural heritage spanning centuries of art, literature, and philosophy.',
        'human_score': 4,
        'llm_score': 5,
        'bias': 'Length Bias - More words rated higher'
    },
    'First Option (A)': {
        'text': 'Option A: Use a database index\nOption B: Cache frequently accessed data',
        'human_score': 4,
        'llm_score': 5,
        'bias': 'Position Bias - First option often preferred'
    },
    'Confident Wrong': {
        'text': 'The Great Wall of China is visible from space. This is a well-established fact that has been confirmed by astronauts.',
        'human_score': 1,
        'llm_score': 4,
        'bias': 'Confidence Bias - Confident tone rated higher even when wrong'
    },
    'Hedged Correct': {
        'text': 'I believe the Great Wall might not be visible from space with the naked eye, though I\'m not entirely certain about the specific conditions.',
        'human_score': 4,
        'llm_score': 3,
        'bias': 'Confidence Bias - Hedged language penalized'
    }
}

output_area3 = widgets.Output()

def show_judge_comparison(sample_name):
    with output_area3:
        clear_output(wait=True)
        
        sample = sample_outputs[sample_name]
        
        print(f"Sample Output: {sample_name}")
        print("=" * 50)
        print(f"\n\"{sample['text']}\"")
        print("\n" + "=" * 50)
        
        # Create comparison chart
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='Human Judge',
            x=['Human Judge'],
            y=[sample['human_score']],
            marker_color='#40B8A6',
            text=[f"{sample['human_score']}/5"],
            textposition='outside'
        ))
        
        fig.add_trace(go.Bar(
            name='LLM Judge',
            x=['LLM Judge'],
            y=[sample['llm_score']],
            marker_color='#1A3D4D',
            text=[f"{sample['llm_score']}/5"],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='Human vs LLM Judge Scores',
            yaxis=dict(range=[0, 6], title='Score (1-5)'),
            showlegend=False,
            height=300
        )
        
        fig.show()
        
        diff = sample['llm_score'] - sample['human_score']
        print(f"\nScore Difference: {diff:+d} points")
        print(f"\nBias Detected: {sample['bias']}")

sample_dropdown = widgets.Dropdown(
    options=list(sample_outputs.keys()),
    value='Concise Answer',
    description='Sample:'
)

display(sample_dropdown, output_area3)
widgets.interactive(show_judge_comparison, sample_name=sample_dropdown)
show_judge_comparison('Concise Answer')

### PM Insight: LLM Judge Limitations

**Known LLM Judge Biases:**

| Bias | What Happens | Mitigation |
|------|--------------|------------|
| Length | Longer = better | Normalize by length |
| Position | First option preferred | Randomize order |
| Self-preference | Prefers own style | Use different model family |
| Confidence | Penalizes hedging | Include factuality check |

> **Key Decision:** Use LLM-as-Judge for speed and scale, but validate against human judgments on a sample set. Expect ~85% agreement on clear cases, ~60% on edge cases.

---

## Demo 4: A/B Test Sample Size Calculator

Before launching an A/B test for an AI feature, you need to know: How many users do I need?

**Try this:** Adjust the parameters to see how sample size requirements change dramatically.

In [ ]:
#@title A/B Test Sample Size Calculator

output_area4 = widgets.Output()

def calculate_sample_size(baseline_rate, expected_lift, confidence_level, power):
    """Calculate required sample size for A/B test"""
    p1 = baseline_rate / 100
    p2 = p1 * (1 + expected_lift / 100)
    
    alpha = 1 - confidence_level / 100
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power / 100)
    
    p_pooled = (p1 + p2) / 2
    
    numerator = (z_alpha * np.sqrt(2 * p_pooled * (1 - p_pooled)) +
                 z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    denominator = (p2 - p1) ** 2
    
    if denominator == 0:
        return float('inf')
    
    return int(np.ceil(numerator / denominator))

def run_calculator(baseline, lift, confidence, power, daily_traffic):
    with output_area4:
        clear_output(wait=True)
        
        sample_size = calculate_sample_size(baseline, lift, confidence, power)
        days_needed = int(np.ceil(sample_size / (daily_traffic / 2))) if daily_traffic > 0 else float('inf')
        
        # Create chart showing sample size vs lift
        lifts = [5, 10, 15, 20, 25, 30]
        sizes = [calculate_sample_size(baseline, l, confidence, power) for l in lifts]
        
        colors = ['#dc2626' if s > 5000 else '#f59e0b' if s > 1000 else '#22c55e' for s in sizes]
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            x=[f"{l}%" for l in lifts],
            y=sizes,
            marker_color=colors,
            text=[f"{s:,}" for s in sizes],
            textposition='outside'
        ))
        
        fig.add_hline(y=sample_size, line_dash="dash", line_color="#40B8A6",
                     annotation_text=f"Your target: {sample_size:,}")
        
        fig.update_layout(
            title=f'Sample Size Required by Expected Lift (Baseline: {baseline}%)',
            xaxis_title='Minimum Detectable Lift',
            yaxis_title='Sample Size per Group',
            height=400
        )
        
        fig.show()
        
        print(f"\n{'='*50}")
        print(f"YOUR A/B TEST PLAN")
        print(f"{'='*50}")
        print(f"\nRequired sample size: {sample_size:,} per group ({sample_size*2:,} total)")
        print(f"At {daily_traffic:,} users/day: ~{days_needed} days to complete")
        print(f"\nConfiguration:")
        print(f"  - Baseline rate: {baseline}%")
        print(f"  - Minimum detectable lift: {lift}%")
        print(f"  - Confidence level: {confidence}%")
        print(f"  - Statistical power: {power}%")
        
        if days_needed > 30:
            print(f"\nWarning: Test will take over a month. Consider:")
            print(f"  - Targeting a larger minimum lift")
            print(f"  - Increasing traffic to the test")
            print(f"  - Accepting lower confidence (90% instead of 95%)")

baseline_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Baseline %:')
lift_slider = widgets.IntSlider(value=10, min=5, max=30, step=5, description='Min Lift %:')
confidence_slider = widgets.IntSlider(value=95, min=90, max=99, step=1, description='Confidence:')
power_slider = widgets.IntSlider(value=80, min=70, max=95, step=5, description='Power:')
traffic_slider = widgets.IntSlider(value=5000, min=500, max=50000, step=500, description='Daily Users:')

ui = widgets.VBox([baseline_slider, lift_slider, confidence_slider, power_slider, traffic_slider])
out = widgets.interactive_output(run_calculator, {
    'baseline': baseline_slider,
    'lift': lift_slider,
    'confidence': confidence_slider,
    'power': power_slider,
    'daily_traffic': traffic_slider
})

display(ui, output_area4)
run_calculator(10, 10, 95, 80, 5000)

### PM Insight: The Sample Size Trap

Notice how sample size **explodes** for small lifts:
- 20% lift: ~200 per group
- 10% lift: ~800 per group
- 5% lift: ~3,000+ per group

**Common A/B testing mistakes:**

1. **Peeking too early** - Checking results before reaching sample size inflates false positives
2. **Ignoring practical significance** - A 2% lift might be statistically significant but not worth the engineering cost
3. **Testing on biased samples** - Weekend traffic behaves differently than weekday
4. **Not accounting for novelty effects** - Users engage more with anything new

> **Key Decision:** Calculate sample size BEFORE starting the test. If you can't reach it in a reasonable time, either target a larger lift or use a different evaluation method.

---

## Demo 5: The Evaluation Pyramid

Different evaluation methods serve different purposes. Use them in combination.

**Try this:** Explore when to use each evaluation level.

In [ ]:
#@title The Evaluation Pyramid

eval_levels = {
    'Unit Tests': {
        'purpose': 'Catch obvious failures before deployment',
        'examples': ['Response not empty', 'JSON parses correctly', 'Latency < 2 seconds'],
        'frequency': 'Every commit',
        'cost': 'Low - automated',
        'coverage': '100% of code paths'
    },
    'Integration Tests': {
        'purpose': 'Verify end-to-end behavior on known inputs',
        'examples': ['Golden set of 100 queries', 'Regression test suite', 'Edge case library'],
        'frequency': 'Every deployment',
        'cost': 'Medium - requires maintenance',
        'coverage': 'Critical paths'
    },
    'LLM-as-Judge': {
        'purpose': 'Scale evaluation to thousands of samples',
        'examples': ['Quality scoring on random sample', 'Comparative ranking', 'Consistency checks'],
        'frequency': 'Weekly or per release',
        'cost': 'Medium - API costs',
        'coverage': 'Representative sample'
    },
    'Human Evaluation': {
        'purpose': 'Ground truth for subjective quality',
        'examples': ['Expert review of edge cases', 'User satisfaction surveys', 'Red team testing'],
        'frequency': 'Monthly or per major release',
        'cost': 'High - human time',
        'coverage': 'Targeted sample'
    },
    'A/B Testing': {
        'purpose': 'Measure real-world impact on users',
        'examples': ['Click-through rate', 'Task completion', 'User retention'],
        'frequency': 'Per feature launch',
        'cost': 'High - requires traffic',
        'coverage': 'Production traffic'
    }
}

output_area5 = widgets.Output()

def show_eval_level(level):
    with output_area5:
        clear_output(wait=True)
        
        info = eval_levels[level]
        
        # Create funnel visualization
        levels = list(eval_levels.keys())
        values = [100, 80, 60, 40, 20]  # Decreasing as we go up
        
        colors = ['#e0f2fe' if l != level else '#40B8A6' for l in levels]
        
        fig = go.Figure(go.Funnel(
            y=levels,
            x=values,
            textinfo="label",
            marker={"color": colors}
        ))
        
        fig.update_layout(
            title='Evaluation Pyramid - Click levels to explore',
            height=400
        )
        
        fig.show()
        
        print(f"\n{'='*50}")
        print(f"{level.upper()}")
        print(f"{'='*50}")
        print(f"\nPurpose: {info['purpose']}")
        print(f"\nExamples:")
        for ex in info['examples']:
            print(f"  - {ex}")
        print(f"\nFrequency: {info['frequency']}")
        print(f"Cost: {info['cost']}")
        print(f"Coverage: {info['coverage']}")

level_dropdown = widgets.Dropdown(
    options=list(eval_levels.keys()),
    value='Unit Tests',
    description='Level:'
)

display(level_dropdown, output_area5)
widgets.interactive(show_eval_level, level=level_dropdown)
show_eval_level('Unit Tests')

### PM Insight: Build the Full Pyramid

Every AI feature needs ALL levels, not just one:

| Stage | What to Check |
|-------|---------------|
| Development | Unit tests catch crashes |
| Pre-deployment | Integration tests verify behavior |
| Post-deployment | LLM judge monitors quality at scale |
| Periodically | Human eval validates LLM judge accuracy |
| New features | A/B tests measure business impact |

> **Key Decision:** Don't skip levels to save time. Meta skipped factuality testing for Galactica and had to shut it down in 3 days.

---

## Final Discussion

**Group Activity:** Design an evaluation plan for this scenario:

> Your company wants to add an AI-powered "smart reply" feature to customer support chat. The AI will suggest 3 response options to agents, who can edit and send them.

Discuss:
1. What are the top 3 evaluation dimensions for this feature?
2. What unit tests would catch obvious failures?
3. How would you set up human evaluation? What rubric?
4. What A/B test metrics would show business value?
5. What failure modes might slip through your evaluation?

---

## Stakeholder Framing: Explaining Evaluation to Leadership

### For Your VP of Engineering:

> "We're implementing a multi-layer evaluation approach: automated tests catch crashes, LLM-based evaluation scales to thousands of samples daily, and human review validates edge cases monthly. This gives us confidence in quality without blocking velocity."

### For Your CFO:

> "Our A/B testing framework ensures we only ship features that show statistically significant improvement. We need 2 weeks of data per test to reach valid conclusions - rushing to results creates false positives that waste engineering resources."

### For Your Chief Product Officer:

> "We learned from Meta's Galactica launch: they tested for fluency but not factuality, and had to shut down after 3 days. Our evaluation framework explicitly tests for the dimensions that matter most for our use case - [accuracy/safety/helpfulness]."

---

## Key Takeaways

1. **What you measure is what you get** - Galactica was fluent but not factual because they only tested fluency
2. **LLM judges are fast but biased** - Use them for scale, validate with humans
3. **Sample size matters** - Calculate before starting, not after
4. **Build the full pyramid** - No single evaluation method is sufficient
5. **Evaluation is ongoing** - Quality can degrade over time without monitoring

---

## Continue Learning

**Interactive Apps (run these for more practice):**
- [LLM Output Evaluator](https://huggingface.co/spaces/axelsirota/llm-output-evaluator) - Score real outputs with LLM judges
- [LLM Judge Comparator](https://huggingface.co/spaces/axelsirota/llm-judge-comparator) - Compare strict vs lenient judges
- [Evaluation Suite Builder](https://huggingface.co/spaces/axelsirota/evaluation-suite-builder) - Design custom rubrics
- [A/B Test Simulator](https://huggingface.co/spaces/axelsirota/ab-test-simulator) - Practice sample size calculations

**Corporate Firewall?** If the HF Spaces links don't work, use the Colab notebooks in the `notebooks/workshop-apps/` folder.

---

*Section 6 Complete - Next: Section 7 - Guardrails*